# Data-Cleaning Approval Workflow (LangGraph HITL)

This notebook implements a human-in-the-loop workflow that:
1. Profiles data quality issues
2. Suggests cleaning actions
3. Pauses for approval
4. Applies only approved transformations


## Human-in-the-Loop Design

Data cleaning can be destructive. The HITL gate prevents accidental data loss.

```text
START
  -> profile_data
  -> suggest_cleaning_actions
  -> [PAUSE before apply_approved_actions]
  -> apply_approved_actions
  -> END
```

Why pause here:
- user validates risk before mutation
- workflow captures explicit approval decision
- transformations are auditable and reproducible


## Node Design and State Passing

Each node writes specific fields for downstream use:
- `profile_data` -> `profile_report`, `rows_before`
- `suggest_cleaning_actions` -> `suggested_actions`
- `apply_approved_actions` -> `cleaning_log`, `rows_after`, `quality_metrics`

Shared state acts as the contract across nodes.


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'pandas', 'numpy'])
print('Dependencies ready')


Dependencies ready


In [2]:
from typing import Dict, List, TypedDict

import numpy as np
import pandas as pd
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

print('Imports successful')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports successful


## Step 2 - Build a Deliberately Messy Dataset

In [3]:
np.random.seed(42)

dirty_df = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10],
    'name': ['Alice', ' Bob ', 'CHARLIE', 'alice', None, 'alice', 'Diana', 'Eve', 'Frank', 'Grace', 'Henry'],
    'email': ['alice@mail.com', 'bob@mail', 'charlie@mail.com', 'alice@mail.com', None, 'alice@mail.com', 'diana@mail.com', 'eve@mail.com', 'frank@mail.com', 'grace@mail.com', 'henry@mail.com'],
    'age': [28, 35, -4, 28, 140, 28, 31, np.nan, 45, 29, 33],
    'salary': [60000, 0, 75000, np.nan, 91000, 91000, 55000, None, 70000, 67000, 65000],
    'department': ['Engineering', 'engineering', 'ENGINEERING', 'Sales', 'sales', 'sales ', 'HR', 'H.R.', 'Marketing', None, 'Marketing'],
})

working_df = dirty_df.copy()

print('Dirty data shape:', dirty_df.shape)
print(dirty_df.head(8).to_string(index=False))


Dirty data shape: (11, 6)
 customer_id    name            email   age  salary  department
           1   Alice   alice@mail.com  28.0 60000.0 Engineering
           2    Bob          bob@mail  35.0     0.0 engineering
           3 CHARLIE charlie@mail.com  -4.0 75000.0 ENGINEERING
           4   alice   alice@mail.com  28.0     NaN       Sales
           5     NaN              NaN 140.0 91000.0       sales
           5   alice   alice@mail.com  28.0 91000.0      sales 
           6   Diana   diana@mail.com  31.0 55000.0          HR
           7     Eve     eve@mail.com   NaN     NaN        H.R.


## Step 3 - Define State Schema

In [4]:
class CleaningState(TypedDict):
    profile_report: str
    suggested_actions: List[Dict[str, str]]
    approval_decision: str
    approval_notes: str
    cleaning_log: List[str]
    rows_before: int
    rows_after: int
    quality_metrics: str

print('State schema ready')


State schema ready


## Step 4 - Node: Profile Data

In [5]:
def profile_data(state: CleaningState) -> dict:
    global working_df
    df = working_df

    missing = df.isnull().sum()
    duplicate_rows = int(df.duplicated().sum())
    invalid_ages = int(((df['age'] < 0) | (df['age'] > 120)).fillna(False).sum())
    zero_salary = int((df['salary'] == 0).fillna(False).sum())

    report_lines = [
        f'Rows={len(df)}, Columns={len(df.columns)}',
        'Missing values by column: ' + str({k: int(v) for k, v in missing.items() if v > 0}),
        f'Exact duplicate rows={duplicate_rows}',
        f'Invalid ages (<0 or >120)={invalid_ages}',
        f'Zero salary rows={zero_salary}',
    ]

    if 'department' in df.columns:
        uniq_raw = df['department'].dropna().nunique()
        uniq_norm = df['department'].dropna().str.strip().str.lower().nunique()
        report_lines.append(f'Department variants due to case/whitespace={int(uniq_raw - uniq_norm)}')

    report = '\n'.join(report_lines)

    print('Node profile_data complete')
    return {'profile_report': report, 'rows_before': len(df)}


## Step 5 - Node: Suggest Cleaning Actions

In [6]:
def suggest_cleaning_actions(state: CleaningState) -> dict:
    actions = [
        {'action': 'trim_strings', 'description': 'Trim whitespace in object columns', 'risk': 'low'},
        {'action': 'normalize_department_case', 'description': 'Standardize department labels', 'risk': 'low'},
        {'action': 'set_invalid_age_to_nan', 'description': 'Replace invalid age values with NaN', 'risk': 'medium'},
        {'action': 'fill_salary_null_or_zero_with_median', 'description': 'Impute salary nulls/zeros using median', 'risk': 'medium'},
        {'action': 'drop_exact_duplicates', 'description': 'Remove exact duplicate rows', 'risk': 'high'},
    ]

    print('Node suggest_cleaning_actions complete with', len(actions), 'proposals')
    return {'suggested_actions': actions}


## Step 6 - Node: Apply Approved Actions

In [7]:
def apply_approved_actions(state: CleaningState) -> dict:
    global working_df
    df = working_df.copy()
    logs = []

    approved_text = (state.get('approval_decision', '') + ' ' + state.get('approval_notes', '')).lower()

    def approved(token: str) -> bool:
        return ('approve all' in approved_text) or (token in approved_text)

    if approved('trim') or approved('whitespace'):
        for col in df.select_dtypes(include=['object']).columns:
            df[col] = df[col].astype(str).str.strip().replace('None', np.nan)
        logs.append('Applied trim_strings to object columns')

    if approved('department') or approved('normalize'):
        if 'department' in df.columns:
            df['department'] = df['department'].replace('H.R.', 'HR')
            df['department'] = df['department'].where(df['department'].isna(), df['department'].str.strip().str.title())
            df['department'] = df['department'].replace('Hr', 'HR')
            logs.append('Normalized department labels')

    if approved('age') or approved('invalid age'):
        mask = ((df['age'] < 0) | (df['age'] > 120)).fillna(False)
        count = int(mask.sum())
        df.loc[mask, 'age'] = np.nan
        logs.append(f'Set {count} invalid ages to NaN')

    if approved('salary') or approved('median'):
        med = float(df['salary'].replace(0, np.nan).median())
        null_before = int(df['salary'].isna().sum())
        zero_before = int((df['salary'] == 0).fillna(False).sum())
        df['salary'] = df['salary'].replace(0, np.nan).fillna(med)
        logs.append(f'Filled {null_before} null and {zero_before} zero salaries with median={med:.2f}')

    if approved('duplicate') or approved('drop exact'):
        before = len(df)
        df = df.drop_duplicates()
        logs.append(f'Removed {before - len(df)} exact duplicate rows')

    working_df = df

    quality = {
        'rows': int(len(df)),
        'missing_total': int(df.isnull().sum().sum()),
        'duplicates': int(df.duplicated().sum()),
        'invalid_age': int(((df['age'] < 0) | (df['age'] > 120)).fillna(False).sum()),
        'zero_salary': int((df['salary'] == 0).fillna(False).sum()),
    }

    print('Node apply_approved_actions complete with', len(logs), 'transformations')
    return {'cleaning_log': logs, 'rows_after': len(df), 'quality_metrics': str(quality)}


## Step 7 - Build Graph With Approval Pause

In [8]:
builder = StateGraph(CleaningState)
builder.add_node('profile_data', profile_data)
builder.add_node('suggest_cleaning_actions', suggest_cleaning_actions)
builder.add_node('apply_approved_actions', apply_approved_actions)

builder.add_edge(START, 'profile_data')
builder.add_edge('profile_data', 'suggest_cleaning_actions')
builder.add_edge('suggest_cleaning_actions', 'apply_approved_actions')
builder.add_edge('apply_approved_actions', END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory, interrupt_before=['apply_approved_actions'])

print('Graph compiled with HITL pause before applying transformations')


Graph compiled with HITL pause before applying transformations


## Approval Gate Explanation

Phase 1: run profiling and action suggestions only.
Phase 2: human reviews proposal list and writes approval decision.
Phase 3: workflow resumes and applies only approved actions.


## Step 8 - Run Phase 1 (Profile + Suggest, then Pause)

In [9]:
config = {'configurable': {'thread_id': 'cleaning-35-thread-1'}}

initial_state = {
    'profile_report': '',
    'suggested_actions': [],
    'approval_decision': '',
    'approval_notes': '',
    'cleaning_log': [],
    'rows_before': 0,
    'rows_after': 0,
    'quality_metrics': '',
}

phase1 = app.invoke(initial_state, config)

print('Workflow paused for approval')
print('\nPROFILE REPORT')
print(phase1['profile_report'])

print('\nSUGGESTED ACTIONS')
for a in phase1['suggested_actions']:
    print('-', a['action'], '| risk=', a['risk'], '|', a['description'])


Node profile_data complete
Node suggest_cleaning_actions complete with 5 proposals
Workflow paused for approval

PROFILE REPORT
Rows=11, Columns=6
Missing values by column: {'name': 1, 'email': 1, 'age': 1, 'salary': 2, 'department': 1}
Exact duplicate rows=0
Invalid ages (<0 or >120)=2
Zero salary rows=1
Department variants due to case/whitespace=4

SUGGESTED ACTIONS
- trim_strings | risk= low | Trim whitespace in object columns
- normalize_department_case | risk= low | Standardize department labels
- set_invalid_age_to_nan | risk= medium | Replace invalid age values with NaN
- fill_salary_null_or_zero_with_median | risk= medium | Impute salary nulls/zeros using median
- drop_exact_duplicates | risk= high | Remove exact duplicate rows


## Step 9 - Human Approval and Resume

In [10]:
approval_decision = 'approved_with_limits'
approval_notes = (
    'Approve trim, department normalization, invalid age handling, salary imputation, and duplicate removal. '
    'Proceed with all suggested actions for this dataset.'
)

app.update_state(config, {'approval_decision': approval_decision, 'approval_notes': approval_notes})
phase2 = app.invoke(None, config)

print('Cleaning completed after approval')
print('Rows:', phase2['rows_before'], '->', phase2['rows_after'])
print('Cleaning log:')
for line in phase2['cleaning_log']:
    print('-', line)
print('Quality metrics:', phase2['quality_metrics'])


Node apply_approved_actions complete with 5 transformations
Cleaning completed after approval
Rows: 11 -> 11
Cleaning log:
- Applied trim_strings to object columns
- Normalized department labels
- Set 2 invalid ages to NaN
- Filled 2 null and 1 zero salaries with median=68500.00
- Removed 0 exact duplicate rows
Quality metrics: {'rows': 11, 'missing_total': 6, 'duplicates': 0, 'invalid_age': 0, 'zero_salary': 0}


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45084\1711424784.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


## Step 10 - Inspect Cleaned Data

In [11]:
print('Cleaned data sample:')
print(working_df.head(10).to_string(index=False))


Cleaned data sample:
 customer_id    name            email  age  salary  department
           1   Alice   alice@mail.com 28.0 60000.0 Engineering
           2     Bob         bob@mail 35.0 68500.0 Engineering
           3 CHARLIE charlie@mail.com  NaN 75000.0 Engineering
           4   alice   alice@mail.com 28.0 68500.0       Sales
           5     NaN              NaN  NaN 91000.0       Sales
           5   alice   alice@mail.com 28.0 91000.0       Sales
           6   Diana   diana@mail.com 31.0 55000.0          HR
           7     Eve     eve@mail.com  NaN 68500.0          HR
           8   Frank   frank@mail.com 45.0 70000.0   Marketing
           9   Grace   grace@mail.com 29.0 67000.0         NaN


## Step 11 - Checkpoint History

In [12]:
history = list(app.get_state_history(config))
print('Checkpoint count:', len(history))
for i, snap in enumerate(history):
    src = 'init'
    if snap.metadata and 'source' in snap.metadata:
        src = snap.metadata['source']
    keys = [k for k, v in snap.values.items() if v not in ('', [], None)]
    print(f'Checkpoint {i}: source={src}, populated_fields={keys}')


Checkpoint count: 6
Checkpoint 0: source=loop, populated_fields=['profile_report', 'suggested_actions', 'approval_decision', 'approval_notes', 'cleaning_log', 'rows_before', 'rows_after', 'quality_metrics']
Checkpoint 1: source=update, populated_fields=['profile_report', 'suggested_actions', 'approval_decision', 'approval_notes', 'rows_before', 'rows_after']
Checkpoint 2: source=loop, populated_fields=['profile_report', 'suggested_actions', 'rows_before', 'rows_after']
Checkpoint 3: source=loop, populated_fields=['profile_report', 'rows_before', 'rows_after']
Checkpoint 4: source=loop, populated_fields=['rows_before', 'rows_after']
Checkpoint 5: source=input, populated_fields=[]


## Key Takeaways

1. HITL approval is critical for risky transformations like row drops.
2. Keeping profile and actions explicit improves auditability.
3. Pause/resume in LangGraph creates safe operational control points.
